# Experiment Design & Sample Validity

本 Notebook 对历史 Billing Page 实验样本进行有效性检查，不计算最终购买、收入或退款效果。

## 1. Experiment Definition

- 实验单位：Session（website_session_id）。当前数据按 Session 记录页面曝光和订单结果。
- Control：Session 首次 Billing 实验曝光为 /billing。
- Treatment：Session 首次 Billing 实验曝光为 /billing-2。
- 窗口：2012-09-10（含）至 2012-11-10（不含）。
- 分组来自实际页面曝光还原，不代表本项目拥有随机化日志。

### Experiment Window Evidence Boundary

当前分析窗口为2012-09-10（含）至2012-11-10（不含）。在这一重构窗口内，`/billing`与`/billing-2`均存在曝光。由于缺少原始实验配置、随机化日志以及窗口外的完整实验运行记录，当前分析不能证明该区间覆盖完整实验生命周期；它应被解释为本项目使用的历史实验复盘窗口。


## 2. Design Limitations

当前没有随机化 assignment 日志和事前实验设计文档，因此无法确认真实随机化单位、分流实现或停止规则。SRM 与协变量平衡只能检查是否发现异常，不能证明随机化完全正确。同一用户还可能通过不同 Session 进入两组。

In [1]:
from pathlib import Path
import pandas as pd

OUTPUT_DIR = Path('../output/tables')
profile = pd.read_csv(OUTPUT_DIR / 'experiment_session_profile.csv', parse_dates=['billing_exposure_at', 'experiment_date'])
assert profile['website_session_id'].is_unique
print(f'Sessions: {len(profile):,}')
print(f'Users: {profile.user_id.nunique():,}')

Sessions: 1,311
Users: 1,301


## 3. Sample Ratio Mismatch Check

In [2]:
srm = pd.read_csv(OUTPUT_DIR / 'experiment_srm_check.csv')
srm_display = srm.copy()
for col in ['control_actual_share', 'treatment_actual_share', 'control_expected_share', 'treatment_expected_share']:
    srm_display[col] = srm_display[col].map(lambda x: f'{x:.2%}')
display(srm_display.T.rename(columns={0: 'Result'}))
print('SRM通过：未发现样本比例失配证据。' if bool(srm.loc[0, 'srm_passed']) else 'SRM未通过：需暂停效果分析并检查分流。')

,Result
control_sessions,657
treatment_sessions,654
control_actual_share,50.11%
treatment_actual_share,49.89%
control_expected_share,50.00%
treatment_expected_share,50.00%
chi_square_statistic,0.006865
p_value,0.933967
srm_passed,True


SRM通过：未发现样本比例失配证据。


SRM 以 50/50 为期望比例，使用卡方拟合优度检验。p-value ≥ 0.05 表示未发现明显比例失配，不等于证明随机化正确。

## 4. Covariate Balance Check

In [3]:
balance_tests = pd.read_csv(OUTPUT_DIR / 'experiment_balance_tests.csv')
balance_details = pd.read_csv(OUTPUT_DIR / 'experiment_balance_summary.csv')
tests_display = balance_tests.copy()
tests_display['p_value'] = tests_display['p_value'].map(lambda x: f'{x:.4f}')
tests_display['bias_corrected_cramers_v'] = tests_display['bias_corrected_cramers_v'].map(lambda x: f'{x:.4f}')
tests_display['max_absolute_share_difference'] = tests_display['max_absolute_share_difference'].map(lambda x: f'{x:.2%}')
display(tests_display)

,covariate,chi_square_statistic,degrees_of_freedom,p_value,bias_corrected_cramers_v,max_absolute_share_difference,balance_passed
0,device_type,1.105312,1,0.2931,0.0089,2.04%,True
1,is_repeat_session,0.147232,1,0.7012,0.0000,0.86%,True
2,utm_source,2.075075,2,0.3543,0.0075,2.24%,True
3,utm_campaign,2.067860,2,0.3556,0.0071,2.23%,True
4,experiment_date,58.787222,60,0.5201,0.0000,1.84%,True


In [4]:
for covariate in balance_tests['covariate']:
    section = balance_details[balance_details['covariate'] == covariate].copy()
    for col in ['control_share', 'treatment_share', 'share_difference_treatment_minus_control']:
        section[col] = section[col].map(lambda x: f'{x:.2%}')
    print(f'\n{covariate}')
    display(section.drop(columns='covariate'))


device_type


,category,control_count,treatment_count,control_share,treatment_share,share_difference_treatment_minus_control
0,desktop,585,569,89.04%,87.00%,-2.04%
1,mobile,72,85,10.96%,13.00%,2.04%



is_repeat_session


,category,control_count,treatment_count,control_share,treatment_share,share_difference_treatment_minus_control
2,0,571,574,86.91%,87.77%,0.86%
3,1,86,80,13.09%,12.23%,-0.86%



utm_source


,category,control_count,treatment_count,control_share,treatment_share,share_difference_treatment_minus_control
4,(missing),87,72,13.24%,11.01%,-2.23%
5,bsearch,141,155,21.46%,23.70%,2.24%
6,gsearch,429,427,65.30%,65.29%,-0.01%



utm_campaign


,category,control_count,treatment_count,control_share,treatment_share,share_difference_treatment_minus_control
7,(missing),87,72,13.24%,11.01%,-2.23%
8,brand,35,42,5.33%,6.42%,1.09%
9,nonbrand,535,540,81.43%,82.57%,1.14%



experiment_date


,category,control_count,treatment_count,control_share,treatment_share,share_difference_treatment_minus_control
10,2012-09-10,10,6,1.52%,0.92%,-0.60%
11,2012-09-11,9,12,1.37%,1.83%,0.46%
12,2012-09-12,15,7,2.28%,1.07%,-1.21%
13,2012-09-13,6,17,0.91%,2.60%,1.69%
14,2012-09-14,11,8,1.67%,1.22%,-0.45%
...,...,...,...,...,...,...
66,2012-11-05,16,9,2.44%,1.38%,-1.06%
67,2012-11-06,16,17,2.44%,2.60%,0.16%
68,2012-11-07,14,18,2.13%,2.75%,0.62%
69,2012-11-08,12,10,1.83%,1.53%,-0.30%


平衡判断同时参考：整体卡方 p-value ≥ 0.05、偏差校正 Cramér’s V < 0.10、最大类别占比差 < 5pp。日期为高基数变量，因此采用偏差校正后的 Cramér’s V，并保留逐日分布明细。

## 5. User Cross-group Sensitivity Check

In [5]:
crossover = pd.read_csv(OUTPUT_DIR / 'experiment_user_crossover_check.csv')
cross_display = crossover.copy()
cross_display['cross_group_user_share'] = cross_display['cross_group_user_share'].map(lambda x: f'{x:.2%}')
display(cross_display.T.rename(columns={0: 'Result'}))
if bool(crossover.loc[0, 'sensitivity_analysis_required']):
    print('存在跨组用户。下一阶段需要报告全样本、排除跨组用户、每用户首次实验Session三套结果。')

,Result
experiment_users,1301
cross_group_users,6
cross_group_user_share,0.46%
sessions_from_cross_group_users,12
sensitivity_analysis_required,True
recommended_sensitivity_analysis,exclude cross-group users and retain first exp...


存在跨组用户。下一阶段需要报告全样本、排除跨组用户、每用户首次实验Session三套结果。


## 6. Validity Conclusion

- Session 级没有双版本曝光。
- SRM 检查通过。
- 五项协变量未发现统计或实践意义上的明显失衡。
- 存在少量用户跨组，因此可以进入指标分析，但必须增加用户跨组敏感性分析。
- 以上结果支持样本具备基本可比性，但不构成随机化机制已被证明的证据。